Load Dataset from Kaggle

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ealaxi/paysim1")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'paysim1' dataset.
Path to dataset files: /kaggle/input/paysim1


Read CSV and Select sample set

In [ ]:
import pandas as pd
df = pd.read_csv("/kaggle/input/paysim1/PS_20174392719_1491204439457_log.csv")
df = df.sample(100000)

In [ ]:
df.head(20)

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
3173238,238,CASH_OUT,58025.33,C1297682471,0.00,0.00,C893024647,129242.81,187268.15,0,0
2417895,202,CASH_IN,176733.81,C1586080729,6472940.41,6649674.22,C709548070,304090.78,127356.97,0,0
2916726,229,CASH_IN,64477.99,C440825665,338502.00,402979.99,C773199339,2145089.99,2080611.99,0,0
2902208,228,CASH_OUT,217940.83,C786161992,0.00,0.00,C1297180200,427192.23,645133.06,0,0
2448311,203,PAYMENT,24958.19,C485740184,0.00,0.00,M1338040683,0.00,0.00,0,0
217130,13,CASH_IN,257764.54,C1637672179,7809037.35,8066801.90,C403062363,1005096.49,742360.81,0,0
3567028,260,PAYMENT,2262.11,C581137425,97108.00,94845.89,M1986690237,0.00,0.00,0,0
4521508,326,CASH_OUT,224402.32,C1997699021,80444.72,0.00,C380770866,366197.85,590600.17,0,0
812143,40,TRANSFER,794266.27,C1698293751,0.00,0.00,C1076515215,1523643.25,2317909.52,0,0
3573205,261,CASH_OUT,14771.85,C1836377223,206837.00,192065.15,C1138600291,0.00,14771.85,0,0


Data Cleaning - Filtering out only needed columns

In [ ]:
df_new = df[['step','type','amount','nameOrig','isFraud']]

In [ ]:
df_new.head(20)

,step,type,amount,nameOrig,isFraud
3173238,238,CASH_OUT,58025.33,C1297682471,0
2417895,202,CASH_IN,176733.81,C1586080729,0
2916726,229,CASH_IN,64477.99,C440825665,0
2902208,228,CASH_OUT,217940.83,C786161992,0
2448311,203,PAYMENT,24958.19,C485740184,0
217130,13,CASH_IN,257764.54,C1637672179,0
3567028,260,PAYMENT,2262.11,C581137425,0
4521508,326,CASH_OUT,224402.32,C1997699021,0
812143,40,TRANSFER,794266.27,C1698293751,0
3573205,261,CASH_OUT,14771.85,C1836377223,0


Check missing values

In [ ]:
df_new.isnull().sum()

,0
step,0
type,0
amount,0
nameOrig,0
isFraud,0


Check for Duplicates

In [ ]:
df_new = df_new.drop_duplicates()

Convert Types

In [ ]:
df_new['amount'] = df_new['amount'].astype(float)

Feature Engineering

In [ ]:
#Create transfer flag
df_new['is_transfer'] = (df_new['type'] == 'TRANSFER').astype(int)

In [ ]:
#Transaction per step
df_new['step_bin'] = df_new['step'] // 24   # daily grouping

In [ ]:
#Coustomer-Level Table
customer_df = df_new.groupby('nameOrig').agg({
    'amount': ['count','mean','max','sum'],
    'isFraud': 'sum',
    'is_transfer': 'mean'
})

In [ ]:
#Rename columns
customer_df.columns = [
    'total_txn','avg_amount','max_amount','total_amount',
    'fraud_count','transfer_ratio'
]

In [ ]:
#Create Thresholds
amount_thresh = customer_df['avg_amount'].quantile(0.95)
txn_thresh = customer_df['total_txn'].quantile(0.90)

In [ ]:
#Risk Scoring
def risk_score(row):
    score = 0

    if row['avg_amount'] > amount_thresh:
        score += 30
    if row['total_txn'] > txn_thresh:
        score += 20
    if row['transfer_ratio'] > 0.6:
        score += 30
    if row['fraud_count'] > 0:
        score += 50

    return score

customer_df['risk_score'] = customer_df.apply(risk_score, axis=1)

In [ ]:
#Label customers
def label(score):
    if score > 70:
        return 'High'
    elif score > 30:
        return 'Medium'
    else:
        return 'Low'

customer_df['risk_label'] = customer_df['risk_score'].apply(label)

In [ ]:
#See results
customer_df.sort_values(by='risk_score', ascending=False).head(10)

,total_txn,avg_amount,max_amount,total_amount,fraud_count,transfer_ratio,risk_score,risk_label
nameOrig,,,,,,,,
C1064157081,1,563149.41,563149.41,563149.41,1,1.0,110,High
C897190163,1,2600970.32,2600970.32,2600970.32,1,1.0,110,High
C1609180237,1,3156421.43,3156421.43,3156421.43,1,1.0,110,High
C415681195,1,936976.86,936976.86,936976.86,1,1.0,110,High
C1058391661,1,2498370.12,2498370.12,2498370.12,1,1.0,110,High
C456316207,1,2642116.95,2642116.95,2642116.95,1,1.0,110,High
C1962597060,1,7188425.25,7188425.25,7188425.25,1,1.0,110,High
C166433413,1,1465173.76,1465173.76,1465173.76,1,1.0,110,High
C162812306,1,3085336.66,3085336.66,3085336.66,1,1.0,110,High


In [ ]:
#Transaction-Level Detection
amount_thresh = df_new['amount'].quantile(0.95)

df_new['is_suspicious'] = (
    (df_new['amount'] > amount_thresh) |
    (df_new['type'] == 'TRANSFER') & (df_new['amount'] > amount_thresh * 0.8)
).astype(int)

In [ ]:
#Add Explanations
def explain(row):
    reasons = []

    if row['avg_amount'] > amount_thresh:
        reasons.append("High average transaction amount")
    if row['total_txn'] > txn_thresh:
        reasons.append("High transaction frequency")
    if row['transfer_ratio'] > 0.6:
        reasons.append("High transfer activity")
    if row['fraud_count'] > 0:
        reasons.append("Past fraud history")

    return ", ".join(reasons)

customer_df['Reason'] = customer_df.apply(explain, axis=1)

In [ ]:
customer_df.sort_values(by='risk_label', ascending=True).head(10)

,total_txn,avg_amount,max_amount,total_amount,fraud_count,transfer_ratio,risk_score,risk_label,reason
nameOrig,,,,,,,,,
C1891204803,1,692743.57,692743.57,692743.57,1,0.0,80,High,"High average transaction amount, Past fraud hi..."
C779464023,1,1459180.14,1459180.14,1459180.14,1,1.0,110,High,"High average transaction amount, High transfer..."
C1366832046,1,945131.71,945131.71,945131.71,1,0.0,80,High,"High average transaction amount, Past fraud hi..."
C379326181,1,46661.03,46661.03,46661.03,1,1.0,80,High,"High transfer activity, Past fraud history"
C166433413,1,1465173.76,1465173.76,1465173.76,1,1.0,110,High,"High average transaction amount, High transfer..."
C897190163,1,2600970.32,2600970.32,2600970.32,1,1.0,110,High,"High average transaction amount, High transfer..."
C1786386487,1,5521483.54,5521483.54,5521483.54,1,0.0,80,High,"High average transaction amount, Past fraud hi..."
C669700766,1,25071.46,25071.46,25071.46,1,1.0,80,High,"High transfer activity, Past fraud history"
C136293883,1,2425164.58,2425164.58,2425164.58,1,0.0,80,High,"High average transaction amount, Past fraud hi..."


Average risk_score for each risk_label

In [ ]:
average_risk_score_by_label = customer_df.groupby('risk_label')['risk_score'].mean().sort_values(ascending=False)
display(average_risk_score_by_label)

,risk_score
risk_label,
High,87.875000
Medium,59.917417
Low,1.658414
